# Theoretical Foundations: Riemannian Geometry for Molecular Transformations

This notebook introduces the mathematical foundations underlying the Neural Thermodynamic Metric (NTM) approach to predicting RBFE calculation difficulty.

## Table of Contents
1. [The RBFE Problem](#1-the-rbfe-problem)
2. [From Euclidean to Riemannian Distance](#2-from-euclidean-to-riemannian-distance)
3. [Metric Tensors](#3-metric-tensors)
4. [Geodesics: Shortest Paths on Curved Spaces](#4-geodesics)
5. [Thermodynamic Interpretation](#5-thermodynamic-interpretation)
6. [Connection to Free Energy Landscapes](#6-connection-to-free-energy-landscapes)

## Important: What This Model Is (and Isn't)

Before proceeding, we must be clear about what the Neural Thermodynamic Metric (NTM) actually computes.

### What NTM Is NOT

**NTM does not compute true thermodynamic path lengths.** True thermodynamic length requires:

1. **Actual MD trajectories** sampling the transformation at each $\lambda$-window
2. **Ensemble averages** of $\langle \partial H / \partial \lambda \rangle$ from those trajectories
3. **Phase space overlap** measurements between adjacent $\lambda$-states

The thermodynamic length in the strict sense is:

$$\mathcal{L}_{thermo} = \int_0^1 \sqrt{\text{Var}_\lambda\left(\frac{\partial H}{\partial \lambda}\right)} \, d\lambda$$

This can only be computed *after* running the expensive simulations we're trying to avoid.

### What NTM Actually Is

**NTM is a learned surrogate/proxy** that predicts transformation difficulty from molecular structure alone:

| True Thermodynamic Approach | NTM Surrogate Approach |
|----------------------------|------------------------|
| Requires MD trajectories | Uses only molecular graphs (SMILES) |
| Computes phase space overlap | Learns from historical FEP outcomes |
| Measures actual sampling variance | Predicts *expected* difficulty |
| Ground truth | Approximation |

### The Approximation We Make

We hypothesize that **structural similarity in a learned embedding space correlates with thermodynamic similarity**. Specifically:

1. A GNN encodes molecules into embeddings $\mathbf{h} \in \mathbb{R}^d$
2. A learned metric $\mathbf{M}$ weights directions by their impact on FEP difficulty
3. Riemannian distance $d_M(\mathbf{h}_A, \mathbf{h}_B)$ approximates expected difficulty

**This is a machine learning prediction, not a physics calculation.**

### When This Approximation Is Valid

The surrogate is most reliable when:
- Training data covers similar chemical space
- Transformations involve similar perturbation types (R-group changes, not scaffold hops)
- The protein target behaves similarly to training examples

### When to Be Cautious

The surrogate may fail when:
- Novel chemotypes not seen in training
- Unusual binding modes or protein flexibility
- Core hopping or large structural changes
- Different solvation environments

### Why "Thermodynamic" in the Name?

We retain "thermodynamic" because:
1. The training signal comes from thermodynamic calculations (FEP uncertainties)
2. The Riemannian geometry framework mirrors thermodynamic length theory
3. The interpretation (difficulty of transformation) is thermodynamically meaningful

But users should understand: **we are learning a proxy, not computing the physics directly.**

## 1. The RBFE Problem

### What is Relative Binding Free Energy?

In drug discovery, we want to know how tightly a small molecule (ligand) binds to a protein target. The **binding free energy** $\Delta G$ quantifies this:

$$\Delta G = -RT \ln K_d$$

where $K_d$ is the dissociation constant.

**Relative Binding Free Energy (RBFE)** compares two ligands:

$$\Delta\Delta G = \Delta G_B - \Delta G_A$$

This tells us: "If I change ligand A to ligand B, how much does binding affinity change?"

### The Computational Challenge

RBFE calculations use **Free Energy Perturbation (FEP)** or **Thermodynamic Integration (TI)**:

$$\Delta\Delta G = \int_0^1 \left\langle \frac{\partial H}{\partial \lambda} \right\rangle_\lambda d\lambda$$

where $\lambda$ is an alchemical parameter that gradually transforms ligand A into ligand B.

**The problem**: Some transformations converge quickly (hours), others require extensive sampling (days). Predicting this *before* running expensive simulations would save enormous computational resources.

## 2. From Euclidean to Riemannian Distance

### The Intuition

Consider two molecules represented as feature vectors $\mathbf{h}_A$ and $\mathbf{h}_B$ in some embedding space.

**Euclidean distance** treats all directions equally:

$$d_E(A, B) = \|\mathbf{h}_B - \mathbf{h}_A\|_2 = \sqrt{\sum_i (h_{B,i} - h_{A,i})^2}$$

But molecular transformations are *not* isotropic! Changing a methyl to an ethyl is very different from changing an amine to a carboxylic acid, even if the Euclidean distances in descriptor space are similar.

### Riemannian Distance

**Riemannian geometry** allows us to define distances that depend on *where* we are and *which direction* we're moving:

$$d_M(A, B) = \sqrt{(\mathbf{h}_B - \mathbf{h}_A)^T \mathbf{M} (\mathbf{h}_B - \mathbf{h}_A)}$$

where $\mathbf{M}$ is a **positive-definite metric tensor** that can:
- Stretch distances in some directions (make those transformations seem "harder")
- Compress distances in other directions (make those transformations seem "easier")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize Euclidean vs Riemannian distance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Euclidean: circular level sets
theta = np.linspace(0, 2*np.pi, 100)
for r in [0.5, 1.0, 1.5, 2.0]:
    axes[0].plot(r*np.cos(theta), r*np.sin(theta), 'b-', alpha=0.6)
axes[0].set_title('Euclidean Distance\n(All directions equal)', fontsize=12)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Riemannian: elliptical level sets
M = np.array([[4, 0], [0, 1]])  # Stretch in x-direction
for r in [0.5, 1.0, 1.5, 2.0]:
    # Ellipse from metric tensor
    x = r * np.cos(theta) / 2  # Scaled by sqrt(eigenvalue)
    y = r * np.sin(theta)
    axes[1].plot(x, y, 'r-', alpha=0.6)
axes[1].set_title('Riemannian Distance\n(Horizontal = harder)', fontsize=12)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The metric tensor M =")
print(M)
print("\nThis means moving in the x-direction costs 4x more 'distance' than moving in y.")

## 3. Metric Tensors

### Definition

A **metric tensor** $\mathbf{M}$ at a point defines an inner product on the tangent space:

$$\langle \mathbf{u}, \mathbf{v} \rangle_M = \mathbf{u}^T \mathbf{M} \mathbf{v}$$

For our purposes, $\mathbf{M}$ is a $d \times d$ symmetric positive-definite matrix where $d$ is the embedding dimension.

### Types of Metric Tensors

**1. Diagonal Metric** (simpler, fewer parameters):

$$\mathbf{M} = \text{diag}(m_1^2, m_2^2, \ldots, m_d^2)$$

Each $m_i^2$ weights the importance of embedding dimension $i$.

**2. Full Metric** (more expressive, captures correlations):

$$\mathbf{M} = \mathbf{L}\mathbf{L}^T$$

where $\mathbf{L}$ is a lower-triangular matrix (Cholesky factorization ensures positive-definiteness).

### Learning the Metric

We learn $\mathbf{M}$ from data by minimizing:

$$\mathcal{L} = \sum_{(A,B)} \left( d_M(A, B) - y_{AB} \right)^2$$

where $y_{AB}$ is the ground truth difficulty (e.g., RBFE uncertainty from FEP calculations).

In [ ]:
import torch
import torch.nn as nn

class LearnableMetric(nn.Module):
    """A learnable Riemannian metric tensor."""
    
    def __init__(self, dim, metric_type='diagonal'):
        super().__init__()
        self.dim = dim
        self.metric_type = metric_type
        
        if metric_type == 'diagonal':
            # Learn log of diagonal elements (ensures positivity)
            self.log_diag = nn.Parameter(torch.zeros(dim))
        else:
            # Learn lower-triangular Cholesky factor
            self.L = nn.Parameter(torch.eye(dim))
    
    def get_metric(self):
        """Return the metric tensor M."""
        if self.metric_type == 'diagonal':
            return torch.diag(torch.exp(self.log_diag))
        else:
            # M = L @ L^T (guaranteed positive definite)
            L = torch.tril(self.L)
            return L @ L.T
    
    def distance(self, h_a, h_b):
        """Compute Riemannian distance between two points."""
        M = self.get_metric()
        diff = h_b - h_a
        d_sq = torch.sum(diff * (diff @ M), dim=-1)
        return torch.sqrt(d_sq + 1e-8)

# Example
metric = LearnableMetric(dim=4, metric_type='diagonal')
print("Initial metric tensor:")
print(metric.get_metric())

# Simulate two molecular embeddings
h_a = torch.randn(1, 4)
h_b = torch.randn(1, 4)

print(f"\nEuclidean distance: {torch.norm(h_b - h_a).item():.4f}")
print(f"Riemannian distance: {metric.distance(h_a, h_b).item():.4f}")

## 4. Geodesics: Shortest Paths on Curved Spaces <a name="4-geodesics"></a>

### What is a Geodesic?

A **geodesic** is the shortest path between two points under a given metric. In Euclidean space, geodesics are straight lines. In Riemannian space, they can be curved.

### Why Geodesics Matter for RBFE

The geodesic from molecule A to molecule B represents the **minimum-difficulty transformation pathway**. If the geodesic is:
- **Nearly straight**: The transformation is straightforward
- **Highly curved**: The optimal path avoids high-difficulty regions

### Computing Geodesics

For a constant metric tensor (our case), the geodesic is still a straight line in the original coordinates, but its **length** is measured differently:

$$L[\gamma] = \int_0^1 \sqrt{\dot{\gamma}(t)^T \mathbf{M} \dot{\gamma}(t)} \, dt$$

For a straight path $\gamma(t) = (1-t)\mathbf{h}_A + t\mathbf{h}_B$:

$$L = \sqrt{(\mathbf{h}_B - \mathbf{h}_A)^T \mathbf{M} (\mathbf{h}_B - \mathbf{h}_A)}$$

### Optimizing Geodesic Paths

In practice, we discretize the path into waypoints and optimize their positions to minimize total path length:

In [ ]:
def compute_geodesic(h_start, h_end, metric, n_points=50, n_iterations=200):
    """
    Compute the geodesic path between two points by optimizing waypoints.
    
    For a constant metric, this will be a straight line, but the method
    generalizes to position-dependent metrics.
    """
    # Initialize path as straight line
    t = torch.linspace(0, 1, n_points).unsqueeze(1)
    path = h_start * (1 - t) + h_end * t
    
    # Make intermediate points learnable
    waypoints = nn.Parameter(path[1:-1].clone())
    optimizer = torch.optim.Adam([waypoints], lr=0.01)
    
    M = metric.get_metric()
    
    for _ in range(n_iterations):
        optimizer.zero_grad()
        
        # Reconstruct full path
        full_path = torch.cat([h_start, waypoints, h_end], dim=0)
        
        # Compute path length under metric
        segments = full_path[1:] - full_path[:-1]
        segment_lengths = torch.sqrt(torch.sum(segments * (segments @ M), dim=1) + 1e-8)
        total_length = segment_lengths.sum()
        
        total_length.backward()
        optimizer.step()
    
    final_path = torch.cat([h_start, waypoints.detach(), h_end], dim=0)
    return final_path, total_length.item()

# Example: compute geodesic
metric = LearnableMetric(dim=2, metric_type='full')
# Set an anisotropic metric
with torch.no_grad():
    metric.L.copy_(torch.tensor([[2.0, 0.0], [0.5, 1.0]]))

h_start = torch.tensor([[0.0, 0.0]])
h_end = torch.tensor([[1.0, 1.0]])

path, length = compute_geodesic(h_start, h_end, metric, n_points=20)

# Visualize
plt.figure(figsize=(8, 6))
path_np = path.numpy()
plt.plot(path_np[:, 0], path_np[:, 1], 'b-o', markersize=4, label='Geodesic')
plt.plot([0, 1], [0, 1], 'r--', label='Straight line (Euclidean)')
plt.scatter([0, 1], [0, 1], s=100, c=['green', 'red'], zorder=5)
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title(f'Geodesic Path (Length = {length:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

## 5. Thermodynamic Interpretation <a name="5-thermodynamic-interpretation"></a>

### Connection to Statistical Mechanics

The Riemannian distance can be interpreted through the lens of **information geometry** and **thermodynamics**:

The **Fisher information metric** on probability distributions:

$$g_{ij}(\theta) = \mathbb{E}\left[ \frac{\partial \log p(x|\theta)}{\partial \theta_i} \frac{\partial \log p(x|\theta)}{\partial \theta_j} \right]$$

measures how "distinguishable" two nearby distributions are.

### For Molecular Transformations

We can interpret our learned metric similarly:

- **High metric weight** in a direction → Small changes in that feature create very different molecules (from an FEP perspective)
- **Low metric weight** → Changes in that direction don't significantly affect sampling difficulty

### Thermodynamic Length

In non-equilibrium thermodynamics, the **thermodynamic length** of a transformation is:

$$\mathcal{L} = \int_0^1 \sqrt{\dot{\lambda}^T g(\lambda) \dot{\lambda}} \, dt$$

This is exactly a Riemannian path length! The metric $g(\lambda)$ encodes the "resistance" to change at each point in parameter space.

**Key insight**: Our learned metric tensor $\mathbf{M}$ approximates this thermodynamic metric in molecular embedding space.

## 6. Connection to Free Energy Landscapes <a name="6-connection-to-free-energy-landscapes"></a>

### Free Energy Surfaces

In molecular simulations, we often visualize **free energy surfaces** $F(\mathbf{x})$ where:

- **Valleys** = stable states (low free energy)
- **Peaks** = unstable transition states (high free energy)
- **Saddle points** = transition pathways

### Our Energy Landscape

The NTM visualizer creates an analogous landscape in embedding space where:

$$E(\mathbf{h}) = \sqrt{(\mathbf{h} - \mathbf{h}_0)^T \mathbf{M} (\mathbf{h} - \mathbf{h}_0)}$$

This represents the "difficulty" of reaching point $\mathbf{h}$ from a reference $\mathbf{h}_0$.

### Interpreting the Visualization

1. **Two wells** at the ligand positions (stable molecular states)
2. **Barrier** between them (transformation difficulty)
3. **Geodesic** follows the minimum energy path through the saddle point
4. **Barrier height** correlates with expected RBFE uncertainty

In [ ]:
# Visualize a double-well energy landscape
from mpl_toolkits.mplot3d import Axes3D

x = np.linspace(-2, 2, 100)
y = np.linspace(-1.5, 1.5, 100)
X, Y = np.meshgrid(x, y)

# Double-well potential with barrier
# Wells at (-1, 0) and (1, 0), barrier at (0, 0)
well_1 = np.exp(-((X + 1)**2 + Y**2) / 0.3)
well_2 = np.exp(-((X - 1)**2 + Y**2) / 0.3)
barrier = 0.8 * np.exp(-(X**2 + Y**2) / 0.4)

# Energy = base + barrier - wells
E = 0.3 * (X**2 + Y**2) + barrier - 1.5 * well_1 - 1.5 * well_2
E = E - E.min()  # Shift minimum to zero

fig = plt.figure(figsize=(14, 5))

# 3D Surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, E, cmap='viridis', alpha=0.9)
ax1.set_xlabel('Reaction Coordinate 1')
ax1.set_ylabel('Reaction Coordinate 2')
ax1.set_zlabel('Free Energy')
ax1.set_title('Free Energy Landscape (3D)')

# Contour
ax2 = fig.add_subplot(122)
cs = ax2.contour(X, Y, E, levels=15, cmap='viridis')
ax2.clabel(cs, inline=True, fontsize=8)
ax2.plot([-1, 1], [0, 0], 'r--', linewidth=2, label='Geodesic (MEP)')
ax2.scatter([-1, 1], [0, 0], s=100, c=['green', 'blue'], zorder=5)
ax2.set_xlabel('Reaction Coordinate 1')
ax2.set_ylabel('Reaction Coordinate 2')
ax2.set_title('Contour Map with Minimum Energy Path')
ax2.legend()

plt.tight_layout()
plt.show()

print("Key features:")
print("  - Two valleys (wells) at ligand positions")
print("  - Barrier in the middle (transition state)")
print("  - Geodesic follows the minimum energy path")

## Summary

### Key Takeaways

1. **RBFE difficulty** varies across molecular transformations—some converge quickly, others don't

2. **Riemannian geometry** provides a natural framework for encoding this variability through a learned metric tensor

3. **The metric tensor** $\mathbf{M}$ weights different directions in embedding space according to transformation difficulty

4. **Geodesic distance** under this metric predicts RBFE uncertainty

5. **The energy landscape** visualization shows valleys (stable states), peaks (barriers), and optimal pathways

### Next Steps

In the next notebook, we'll see how to:
- Build the Graph Neural Network encoder
- Learn the metric tensor from RBFE data
- Implement the full Neural Thermodynamic Metric model